In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings

c:\Users\ravir\Desktop\Desktop\GenAI Projects\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("test.txt", "r") as f:
    content = f.read()

fixed_content = content.replace(". ", ".\n")

with open("test.txt", "w") as f:
    f.write(fixed_content)

print("Done!")

Done!


**1.Loading .txt file**

In [3]:
loader= TextLoader("test.txt")
documents= loader.load()
documents

[Document(metadata={'source': 'test.txt'}, page_content='Mike and Morris lived in the same village.\nWhile Morris owned the largest jewelry shop in the village, Mike was a poor farmer.\nBoth had large families with many sons, daughters-in-law and grandchildren.\nOne fine day, Mike, tired of not being able to feed his family, decided to leave the village and move to the city where he was certain to earn enough to feed everyone.\nAlong with his family, he left the village for the city.\nAt night, they stopped under a large tree.\nThere was a stream running nearby where they could freshen up themselves.\nHe told his sons to clear the area below the tree, he told his wife to fetch water and he instructed his daughters-in-law to make up the fire and started cutting wood from the tree himself.\nThey didn\'t know that in the branches of the tree, there was a thief hiding.\nHe watched as Mike\'s family worked together and also noticed that they had nothing to cook.\nMike\'s wife also thought t

**2.Splitting dataset into chunks**

In [4]:
text_splitter= CharacterTextSplitter(chunk_size=200, chunk_overlap=20,separator="\n",)
split= text_splitter.split_documents(documents)
split

[Document(metadata={'source': 'test.txt'}, page_content='Mike and Morris lived in the same village.\nWhile Morris owned the largest jewelry shop in the village, Mike was a poor farmer.'),
 Document(metadata={'source': 'test.txt'}, page_content='Both had large families with many sons, daughters-in-law and grandchildren.'),
 Document(metadata={'source': 'test.txt'}, page_content='One fine day, Mike, tired of not being able to feed his family, decided to leave the village and move to the city where he was certain to earn enough to feed everyone.'),
 Document(metadata={'source': 'test.txt'}, page_content='Along with his family, he left the village for the city.\nAt night, they stopped under a large tree.\nThere was a stream running nearby where they could freshen up themselves.'),
 Document(metadata={'source': 'test.txt'}, page_content='He told his sons to clear the area below the tree, he told his wife to fetch water and he instructed his daughters-in-law to make up the fire and started c

In [5]:
len(split)

17

**4.Embeddings and VectorDB**

In [7]:
embeddings= OllamaEmbeddings(model="gemma:2b")
vectordb= FAISS.from_documents(split,embeddings)
vectordb

C:\Users\ravir\AppData\Local\Temp\ipykernel_10808\2076663907.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings= OllamaEmbeddings(model="gemma:2b")


**5.Query and result extraction**

In [8]:
query= "How did the fellow villagers react to Mike getting rich overnight?"
result= vectordb.similarity_search(query)
result

[Document(id='99f8d561-99e9-490d-8113-06f3d3c39211', metadata={'source': 'test.txt'}, page_content='The family gathered all their belongings and returned to the village.\nThere was great excitement when they told everyone how they got rich.'),
 Document(id='5f3e7e04-7fce-4211-80dc-557a43579cfe', metadata={'source': 'test.txt'}, page_content='The thief stole everything they had and Morris and his family had to return to the village empty handed, having lost all their valuables that they had taken with them.'),
 Document(id='14a5cf3e-554d-487f-b4de-4768613fd2d4', metadata={'source': 'test.txt'}, page_content='Mike and Morris lived in the same village.\nWhile Morris owned the largest jewelry shop in the village, Mike was a poor farmer.'),
 Document(id='aa994066-77cc-4ccb-a433-9ca3b698f0d6', metadata={'source': 'test.txt'}, page_content='He told his sons to clear the area below the tree, he told his wife to fetch water and he instructed his daughters-in-law to make up the fire and started 

In [9]:
result[0].page_content

'The family gathered all their belongings and returned to the village.\nThere was great excitement when they told everyone how they got rich.'

**We can also generate answers using Retreiver**

In [10]:
retriever= vectordb.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

'The family gathered all their belongings and returned to the village.\nThere was great excitement when they told everyone how they got rich.'

In [11]:
## Saving in my local worspace

vectordb.save_local("faiss_idx")


In [12]:
new_db= FAISS.load_local("faiss_idx", embeddings, allow_dangerous_deserialization=True)

In [14]:
result= new_db.similarity_search(query)
result

[Document(id='99f8d561-99e9-490d-8113-06f3d3c39211', metadata={'source': 'test.txt'}, page_content='The family gathered all their belongings and returned to the village.\nThere was great excitement when they told everyone how they got rich.'),
 Document(id='5f3e7e04-7fce-4211-80dc-557a43579cfe', metadata={'source': 'test.txt'}, page_content='The thief stole everything they had and Morris and his family had to return to the village empty handed, having lost all their valuables that they had taken with them.'),
 Document(id='14a5cf3e-554d-487f-b4de-4768613fd2d4', metadata={'source': 'test.txt'}, page_content='Mike and Morris lived in the same village.\nWhile Morris owned the largest jewelry shop in the village, Mike was a poor farmer.'),
 Document(id='aa994066-77cc-4ccb-a433-9ca3b698f0d6', metadata={'source': 'test.txt'}, page_content='He told his sons to clear the area below the tree, he told his wife to fetch water and he instructed his daughters-in-law to make up the fire and started 